# HPC for AI: spotting fractures on Isambard-AI

In this notebook you'll train a computer vision model to spot fractures in X-ray images on the Isambard-AI supercomputer, first on one GPU and then on four.

**How to use this notebook**

- Run the cells in order with **Shift + Enter**. Wait for each cell to finish before you run the next one: the `[*]` to the left of the cell changes to a number when it's done.
- The kernel name in the top right should say **Python (HPC for AI part 1)**. If it doesn't, click on it and choose that kernel.
- The course notes go through each step in more detail, and have an exercise for each section.

**Data:** FracAtlas, Abedeen *et al.* (2023), *Scientific Data* 10, 521. [doi:10.1038/s41597-023-02432-4](https://doi.org/10.1038/s41597-023-02432-4). Licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

## 1. Getting started

### Load the libraries

This loads all the libraries we'll use in the notebook. The compute nodes on Isambard-AI don't have internet access, so we also tell the Ultralytics library not to try downloading anything.

In [ ]:
import os
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Image as ShowImage
from IPython.display import display
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split

# Isambard-AI's compute nodes can't reach the internet, so tell Ultralytics
# not to try downloading anything. This must be set before importing it.
os.environ["YOLO_OFFLINE"] = "1"

# Lets training give the same result each time. It has to be set before the
# GPU is first used, which is why it's here at the top.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
from ultralytics import YOLO

### Which GPUs do we have?

You asked for 4 GH200s when you started this session, so PyTorch should be able to see four GPUs.

In [ ]:
print("PyTorch version:", torch.__version__)
print("CUDA available: ", torch.cuda.is_available())
print("Number of GPUs: ", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

### Your first GPU job

Most of the work in training a neural network is multiplying large grids of numbers together. This cell times one large matrix multiplication on the CPU and then on a GPU.

In [ ]:
size = 8192
a = torch.randn(size, size)
b = torch.randn(size, size)

# On the CPU
start = time.perf_counter()
a @ b
cpu_time = time.perf_counter() - start

# On GPU 0
a_gpu = a.to("cuda")
b_gpu = b.to("cuda")
a_gpu @ b_gpu                # warm-up: the first call includes one-off set-up costs
torch.cuda.synchronize()

start = time.perf_counter()
a_gpu @ b_gpu
torch.cuda.synchronize()     # wait for the GPU to finish before stopping the clock
gpu_time = time.perf_counter() - start

print(f"CPU: {cpu_time:.3f} s")
print(f"GPU: {gpu_time:.4f} s")
print(f"The GPU was {cpu_time / gpu_time:.0f}x faster")

### Where things live

The dataset and model weights are in a shared project folder that the instructors have set up. You can read them but you can't change them.

Anything you create goes in your own scratch folder, which is the place for working files like these.

In [ ]:
# Shared course files (read-only for you)
COURSE_DIR = Path(os.environ.get("PROJECTDIR", "/projects/TODO_PROJECT")) / "hpc-ai-1"
DATA_DIR = COURSE_DIR / "FracAtlas"
WEIGHTS = COURSE_DIR / "weights" / "yolov8n-cls.pt"

# Your own working folder
WORK_DIR = Path(os.environ.get("SCRATCHDIR", Path.home())) / "hpc-ai-1"
SPLIT_DIR = WORK_DIR / "fracatlas-split"
RUNS_DIR = WORK_DIR / "runs"
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset found:     ", DATA_DIR.exists(), "-", DATA_DIR)
print("Model weights found:", WEIGHTS.exists(), "-", WEIGHTS)
print("Your work folder:  ", WORK_DIR)

## 2. Meet FracAtlas

FracAtlas has 4,083 X-ray images of hands, legs, hips and shoulders from three hospitals in Bangladesh. Each image is labelled as fractured or not fractured.

### Load the labels

`dataset.csv` has one row for each image. The `fractured` column is the label we want to predict (1 = fractured, 0 = not fractured). The other columns say which part of the body is shown, which angle it was taken from, and whether there's any surgical hardware, such as plates or screws.

In [ ]:
df = pd.read_csv(DATA_DIR / "dataset.csv")

print(df.shape)
df.head()

### How balanced are the classes?

In [ ]:
counts = df["fractured"].value_counts().rename({0: "Not fractured", 1: "Fractured"})
print(counts)
print(f"\n{df['fractured'].mean():.1%} of images show a fracture")

counts.plot(kind="bar", color=["#b895ce", "#3d0251"], rot=0, title="FracAtlas: images per class")
plt.ylabel("Number of images")
plt.show()

### Find the image files

The images are in two folders, `images/Fractured` and `images/Non_fractured`. This adds the full path of each image to the table.

In [ ]:
def image_path(row):
    folder = "Fractured" if row["fractured"] == 1 else "Non_fractured"
    return DATA_DIR / "images" / folder / row["image_id"]


df["path"] = df.apply(image_path, axis=1)

print("All image files found:", df["path"].map(Path.exists).all())

### Look at some examples

Can you spot the fractures? Some of them are quite hard to see.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for row_axes, label, title in zip(axes, [1, 0], ["Fractured", "Not fractured"]):
    sample = df[df["fractured"] == label].sample(4, random_state=1)
    for ax, (_, row) in zip(row_axes, sample.iterrows()):
        ax.imshow(Image.open(row["path"]), cmap="gray")
        ax.set_title(f"{title}\n{row['image_id']}", fontsize=9)
        ax.axis("off")

plt.tight_layout()
plt.show()

### How big are the images?

The images are all different sizes. The model will shrink them all to the same size when it loads them.

In [ ]:
sizes = [Image.open(p).size for p in df["path"].sample(200, random_state=1)]
sizes = pd.DataFrame(sizes, columns=["width", "height"])
sizes.describe().round()

### Fractures by body region

In [ ]:
regions = ["hand", "leg", "hip", "shoulder"]

region_stats = pd.DataFrame(
    {
        "images": [df[r].sum() for r in regions],
        "fractured": [df.loc[df[r] == 1, "fractured"].sum() for r in regions],
    },
    index=regions,
)
region_stats["% fractured"] = (100 * region_stats["fractured"] / region_stats["images"]).round(1)
region_stats

### Split into training, validation and test sets

- The model learns from the **training** set (70%).
- We check how it's doing on the **validation** set (15%) after each epoch.
- We keep the **test** set (15%) aside until the end, to see how well the model does on images it has never seen.

`stratify` keeps the same proportion of fractured images in each set.

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df["fractured"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["fractured"], random_state=42
)

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5}: {len(part):4} images, {part['fractured'].mean():.1%} fractured")

### Arrange the images for the model

The model expects a folder for each set, with a subfolder for each class:

```
fracatlas-split/
├── train/
│   ├── fractured/
│   └── not_fractured/
├── val/
└── test/
```

Rather than copying all 4,083 images, we make *symbolic links*, which are small files that point to the original images in the shared folder. This is quick and takes up almost no space.

In [ ]:
CLASS_NAMES = {1: "fractured", 0: "not_fractured"}

if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)  # start fresh if you re-run this cell (removes the links, not the images)

for split, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    for _, row in part.iterrows():
        link = SPLIT_DIR / split / CLASS_NAMES[row["fractured"]] / row["image_id"]
        link.parent.mkdir(parents=True, exist_ok=True)
        link.symlink_to(row["path"])

for split in ["train", "val", "test"]:
    for class_dir in sorted((SPLIT_DIR / split).iterdir()):
        print(f"{split}/{class_dir.name}: {len(list(class_dir.iterdir()))} images")

## 3. Training on one GPU

We'll use YOLOv8n-cls, a small image classification model from the Ultralytics library. It has already been trained on over a million everyday photos (the ImageNet dataset), so we only need to *fine-tune* it on our X-rays.

### Training settings

- `EPOCHS` is the number of times the model sees every training image.
- `IMG_SIZE` is the size each image is shrunk to, in pixels.
- `BATCH` is the number of images the model looks at before it updates its weights.
- `WORKERS` is the number of CPU processes that load images in the background, so the GPU doesn't have to wait for them.

In [ ]:
EPOCHS = 10      # TODO (instructors): tune after the dry run so each run takes about 10 minutes
IMG_SIZE = 224
BATCH = 64
WORKERS = 8

### Train

Before you run this cell, open the GPU dashboard (the GPU icon in the left-hand sidebar) so you can watch the GPU while the model trains.

In [ ]:
model = YOLO(WEIGHTS)

start = time.perf_counter()
model.train(
    data=str(SPLIT_DIR),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    workers=WORKERS,
    device=0,             # one GPU
    project=str(RUNS_DIR),
    name="1gpu",
    exist_ok=True,
    seed=42,
)
time_1gpu = time.perf_counter() - start

print(f"Training on 1 GPU took {time_1gpu / 60:.1f} minutes")

### How did training go?

Ultralytics saves some plots of the training run. The loss should go down and the validation accuracy should go up.

In [ ]:
run_1gpu = RUNS_DIR / "1gpu"

for plot in ["results.png", "confusion_matrix_normalized.png"]:
    if (run_1gpu / plot).exists():
        display(ShowImage(filename=str(run_1gpu / plot), width=800))

### Test the model on unseen images

Now we load the best version of the model from training and ask it to classify every image in the test set.

In [ ]:
def evaluate(weights):
    """Classify every test image and return the true and predicted labels."""
    model = YOLO(weights)
    true, pred = [], []
    test_images = str(SPLIT_DIR / "test" / "*" / "*.jpg")  # every image in every class folder
    for result in model.predict(test_images, imgsz=IMG_SIZE, stream=True, verbose=False):
        true.append(Path(result.path).parent.name)  # the folder name is the true label
        pred.append(result.names[result.probs.top1])  # the class the model thinks is most likely
    return true, pred


true, pred = evaluate(run_1gpu / "weights" / "best.pt")

print(classification_report(true, pred, digits=3))
ConfusionMatrixDisplay.from_predictions(true, pred, cmap="Purples")
plt.title("1 GPU: test set")
plt.show()

### Is that accuracy any good?

Most of the images aren't fractured, so a "model" that always said "not fractured" would get a fairly high score.

In [ ]:
baseline = (test_df["fractured"] == 0).mean()
print(f"Always guessing 'not fractured' would be {baseline:.1%} accurate")

The **recall** for the `fractured` class (also called *sensitivity*) tells us what proportion of the real fractures the model found. If we wanted to use the model to decide which X-rays to look at first, this would be the most important score, because missing a fracture is worse than a false alarm.

## 4. Scaling to four GPUs

Now we'll train the same model on all four GPUs. Each GPU gets its own copy of the model and a quarter of each batch. After each batch, the GPUs share what they've learned so that all four copies stay the same. This is called *data parallelism*.

Ultralytics does this using PyTorch's Distributed Data Parallel (DDP), which starts a separate Python process for each GPU. Look out for the line starting `DDP:` in the output.

Keep an eye on the GPU dashboard again. This time all four GPUs should be working.

In [ ]:
model = YOLO(WEIGHTS)

start = time.perf_counter()
model.train(
    data=str(SPLIT_DIR),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,          # the total batch is shared out: 64 / 4 = 16 images per GPU
    workers=WORKERS,
    device=[0, 1, 2, 3],  # four GPUs
    project=str(RUNS_DIR),
    name="4gpu",
    exist_ok=True,
    seed=42,
)
time_4gpu = time.perf_counter() - start

print(f"Training on 4 GPUs took {time_4gpu / 60:.1f} minutes")

### Was it faster?

In [ ]:
timings = pd.DataFrame(
    {"minutes": [time_1gpu / 60, time_4gpu / 60]},
    index=["1 GPU", "4 GPUs"],
)
timings["speed-up"] = (time_1gpu / 60 / timings["minutes"]).round(2)
print(timings.round(2))

timings["minutes"].plot(kind="barh", color=["#b895ce", "#3d0251"], title="Training time")
plt.xlabel("Minutes")
plt.show()

### Is the model just as good?

In [ ]:
true, pred = evaluate(RUNS_DIR / "4gpu" / "weights" / "best.pt")

print(classification_report(true, pred, digits=3))
ConfusionMatrixDisplay.from_predictions(true, pred, cmap="Purples")
plt.title("4 GPUs: test set")
plt.show()

### Experiment: give each GPU more work

In the last run, each GPU only got 16 images per batch, so they spent a lot of their time waiting to share results. Here we make the total batch four times bigger, so each GPU gets 64 images per batch, the same as the single GPU had before.

In [ ]:
model = YOLO(WEIGHTS)

start = time.perf_counter()
model.train(
    data=str(SPLIT_DIR),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH * 4,      # 256 / 4 = 64 images per GPU
    workers=WORKERS,
    device=[0, 1, 2, 3],
    project=str(RUNS_DIR),
    name="4gpu-bigbatch",
    exist_ok=True,
    seed=42,
)
time_4gpu_big = time.perf_counter() - start

timings.loc["4 GPUs, 4x batch", "minutes"] = time_4gpu_big / 60
timings["speed-up"] = (time_1gpu / 60 / timings["minutes"]).round(2)
print(timings.round(2))

## 5. Wrapping up

### What did we make?

Your results are in your scratch folder. Scratch isn't backed up, so copy anything you want to keep.

In [ ]:
for run in sorted(RUNS_DIR.iterdir()):
    print(run.name, "->", run / "weights" / "best.pt")

### Before you go

Your session keeps the GPUs reserved until it runs out of time, even if you aren't using them. When you've finished:

1. Save this notebook (**File → Save Notebook**).
2. Go to **File → Hub Control Panel** and click **Stop My Server**.

This lets other people use the GPUs.